# Severity prediction

Comparing Logistic Regression, Decision Tree, Random Forrest, and Gradient Boosting

In [ ]:
# 1. SETUP

import os
import re
import sys
import json
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import Normalizer, OneHotEncoder
from sklearn.cluster import MiniBatchKMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

import scipy.sparse as sp

warnings.filterwarnings("ignore")

print("Python executable:", sys.executable)
print("Python version:", sys.version.split()[0])
print("Pandas version:", pd.__version__)

Python executable: /Users/ariellerothman/Desktop/Sperm Cell Analysis/.venv/bin/python
Python version: 3.12.10
Pandas version: 2.2.3


In [2]:
# Load pre-imputed dataset (instead of re-running imputation)
imputed_data_path = '/Users/ariellerothman/Desktop/Capstone/Outputs/df_clean_imputed.csv'

if os.path.exists(imputed_data_path):
    df_clean = pd.read_csv(imputed_data_path, low_memory=False)
    print(f"✓ Loaded pre-imputed dataset from: {imputed_data_path}")
    print(f"  Shape: {df_clean.shape}")
else:
    print(f"⚠ Imputed dataset not found. Run imputation code first.")

✓ Loaded pre-imputed dataset from: /Users/ariellerothman/Desktop/Capstone/Outputs/df_clean_imputed.csv
  Shape: (986096, 42)


In [ ]:
# RULE-BASED COMORBIDITY INDICATORS 

import os
import re
import numpy as np
import pandas as pd

OUTDIR = "/Users/ariellerothman/Desktop/Capstone/Outputs"
os.makedirs(OUTDIR, exist_ok=True)

# Update this to your dataframe name if needed:
# df_clean = <your cleaned case-level dataframe>

FREE_TEXT_FIELDS = ["HISTORY", "CUR_ILL", "ALLERGIES", "PRIOR_VAX", "LAB_DATA", "OTHER_MEDS"]

# Terms that should be treated as "no information" (so they become missing)
NULL_TOKENS = {
    "none", "no", "n", "na", "n/a", "unknown", "unk", "none known", "not known",
    "nka", "nkda", "no known allergies", "no known drug allergies",
    "denies", "denies allergies", "no allergies", "nil", "negative"
}

_keep = re.compile(r"[^a-z0-9\s\-\/]")
_ws = re.compile(r"\s+")

def normalize_text(x) -> str:
    """Normalize free-text: lowercase, remove urls/emails/specials, collapse whitespace,
    and convert null-like tokens to empty string."""
    if pd.isna(x):
        return ""
    x = str(x).strip().lower()
    if not x:
        return ""

    # Common "null-like" entries
    if x in NULL_TOKENS:
        return ""

    # remove urls/emails
    x = re.sub(r"http\S+|www\S+|https\S+", " ", x)
    x = re.sub(r"\S+@\S+", " ", x)

    # keep alnum + spaces + hyphen + slash
    x = _keep.sub(" ", x)
    x = _ws.sub(" ", x).strip()

    # If after normalization it's basically a null token, drop it
    if x in NULL_TOKENS:
        return ""

    return x

# ---------- UPDATED CATEGORY MAP ----------
# Notes:
# - Add categories where your "other" bucket showed lots of valid clinical signal:
#   hypothyroidism/thyroid, arthritis/OA, migraine, depression/anxiety, GERD, edema/lymphedema.
# - Keep patterns conservative using word boundaries (\b) to reduce false positives.
CATEGORY_MAP = {
    "CUR_ILL": {
        "acute_infection": [
            r"\bcovid\b", r"\bsars[-\s]?cov[-\s]?2\b", r"\binfluenza\b", r"\bflu\b",
            r"\binfection\b", r"\bpneumonia\b", r"\bbronchitis\b", r"\bviral\b", r"\bbacterial\b"
        ],
        "chronic_cardiovascular": [
            r"\bhypertension\b", r"\bhtn\b", r"\bcad\b", r"\bcoronary\b", r"\bafib\b",
            r"\bheart failure\b", r"\bchf\b", r"\barrhythmia\b", r"\bmi\b", r"\bmyocardial infarction\b"
        ],
        "chronic_respiratory": [
            r"\basthma\b", r"\bcopd\b", r"\bemphysema\b", r"\bchronic obstructive\b",
            r"\bsleep apnea\b", r"\bosa\b"
        ],
        "metabolic_endocrine": [
            r"\bdiabetes\b", r"\bdm\b", r"\bdm2\b", r"\btype\s?2\b", r"\bobesity\b",
            r"\bhyperlipid\w*\b", r"\bdyslipid\w*\b"
        ],
        # NEW: thyroid/endocrine explicitly (often shows up in your OTHER examples)
        "endocrine_thyroid": [
            r"\bthyroid\b", r"\bhypothyroid\w*\b", r"\bhyperthyroid\w*\b",
            r"\bhashimoto\w*\b", r"\bgraves\b"
        ],
        "autoimmune_inflammatory": [
            r"\bautoimmune\b", r"\blupus\b", r"\bsle\b", r"\bra\b", r"\brheumatoid arthritis\b",
            r"\bmultiple sclerosis\b", r"\bms\b", r"\bibs\b", r"\bibd\b", r"\bceliac\b"
        ],
        "neurologic": [
            r"\bseizure\b", r"\bepilepsy\b", r"\bstroke\b", r"\btia\b", r"\bparkinson\w*\b",
            r"\bneuropath\w*\b", r"\bdementia\b", r"\balzheimer\w*\b"
        ],
        # NEW: migraine/headache bucket (showed up in your examples)
        "neuro_headache": [
            r"\bmigraine\w*\b", r"\bheadache\w*\b", r"\bcluster headache\b"
        ],
        "cancer_immunocompromised": [
            r"\bcancer\b", r"\bmalignan\w*\b", r"\bchemotherap\w*\b", r"\bradiation\b",
            r"\btransplant\b", r"\bhiv\b", r"\baids\b",
            r"\bimmunosupp\w*\b", r"\bimmunocompromis\w*\b"
        ],
        # NEW (optional): edema/lymphedema bucket (shows in your examples)
        "lymphatic_edema": [
            r"\blymph(edema|oedema)\b", r"\bedema\b", r"\bswelling\b"
        ],
    },

    "HISTORY": {
        "cardiovascular": [
            r"\bhtn\b", r"\bhypertension\b", r"\bcad\b", r"\bcoronary\b", r"\bafib\b",
            r"\bheart failure\b", r"\bchf\b", r"\bmi\b", r"\bmyocardial infarction\b",
            r"\bvalve\b", r"\baortic stenosis\b"
        ],
        "respiratory": [
            r"\basthma\b", r"\bcopd\b", r"\bemphysema\b", r"\bchronic obstructive\b",
            r"\bosa\b", r"\bsleep apnea\b"
        ],
        "metabolic": [
            r"\bdiabetes\b", r"\bdm\b", r"\bdm2\b", r"\bobesity\b",
            r"\bhyperlipid\w*\b", r"\bdyslipid\w*\b"
        ],
        # NEW: thyroid/endocrine (common in HISTORY OTHER)
        "endocrine_thyroid": [
            r"\bthyroid\b", r"\bhypothyroid\w*\b", r"\bhyperthyroid\w*\b",
            r"\bhashimoto\w*\b", r"\bgraves\b"
        ],
        # NEW: musculoskeletal (arthritis/OA showed in your OTHER samples)
        "musculoskeletal": [
            r"\barthritis\b", r"\boveruse\b", r"\bosteoarthritis\b", r"\boa\b",
            r"\bgout\b", r"\bfibromyalgia\b", r"\bchronic pain\b"
        ],
        # NEW: mental health (often appears in HISTORY strings)
        "mental_health": [
            r"\bdepress\w*\b", r"\banxiety\b", r"\bptsd\b", r"\bbipolar\b",
            r"\bschizo\w*\b"
        ],
        # NEW: GI (GERD appears a lot)
        "gastrointestinal": [
            r"\bgerd\b", r"\bacid reflux\b", r"\bgastroesophageal\b",
            r"\bibs\b", r"\bconstipat\w*\b", r"\bdiverticul\w*\b"
        ],
        "kidney": [
            r"\bckd\b", r"\bchronic kidney\b", r"\brenal\b", r"\bdialysis\b",
            r"\brenal failure\b", r"\bkidney transplant\b", r"\btransplant\b"
        ],
        "clotting": [
            r"\bdvt\b", r"\bpe\b", r"\bthromb\w*\b", r"\bclot\b",
            r"\bembol\w*\b", r"\bantiphospholipid\b"
        ],
        "autoimmune": [
            r"\bautoimmune\b", r"\blupus\b", r"\bsle\b", r"\bra\b",
            r"\bms\b", r"\bmultiple sclerosis\b"
        ],
        "cancer_immunocompromised": [
            r"\bcancer\b", r"\bmalignan\w*\b", r"\bchemotherap\w*\b", r"\bradiation\b",
            r"\btransplant\b", r"\bhiv\b", r"\baids\b", r"\bimmunosupp\w*\b"
        ],
        "neurologic": [
            r"\bstroke\b", r"\btia\b", r"\bseizure\b", r"\bepilepsy\b",
            r"\bmigraine\w*\b", r"\bneuropath\w*\b", r"\bparkinson\w*\b"
        ],
        # NEW (optional): lymphatic/edema
        "lymphatic_edema": [
            r"\blymph(edema|oedema)\b", r"\bedema\b"
        ],
    },

    "ALLERGIES": {
        "drug_allergy": [
            r"\bpenicillin\b", r"\bsulfa\b", r"\bnsaid\b", r"\baspirin\b",
            r"\bcephalosporin\b", r"\bceclor\b", r"\bkeflex\b", r"\bdoxycycline\b"
        ],
        "food_allergy": [
            r"\bpeanut\b", r"\bnut\b", r"\begg\b", r"\bmilk\b", r"\bshellfish\b", r"\bgarlic\b"
        ],
        "latex_other_contact": [
            r"\blatex\b", r"\bdetergent\b", r"\bfragrance\b"
        ],
        "anaphylaxis_history": [
            r"\banaphylax\w*\b", r"\bepi[-\s]?pen\b", r"\bepinephrine\b"
        ],
    },

    "PRIOR_VAX": {
        "prior_covid_vax": [r"\bcovid\b", r"\bpfizer\b", r"\bmoderna\b", r"\bjanssen\b", r"\bnovavax\b"],
        "prior_flu_vax": [r"\bflu\b", r"\binfluenza\b"],
        # broaden common vaccines
        "prior_other_vax": [
            r"\bshingrix\b", r"\bshingles\b", r"\btetanus\b", r"\brabies\b",
            r"\bh1n1\b", r"\bmmr\b", r"\bhepatitis\b", r"\bvaricella\b"
        ],
        "prior_vax_reaction": [r"\breaction\b", r"\ballergic\b", r"\banaphylax\w*\b", r"\bside effect\b", r"\bsyncope\b"],
    },

    "LAB_DATA": {
        "cardiac_markers": [r"\btroponin\b", r"\bbnp\b", r"\bck[-\s]?mb\b"],
        "coagulation": [r"\bd[-\s]?dimer\b", r"\binr\b", r"\bptt\b", r"\bfibrinogen\b"],
        "inflammation": [r"\bcrp\b", r"\besr\b", r"\bferritin\b"],
        "cbc": [r"\bwbc\b", r"\bplatelet\w*\b", r"\bhemoglobin\b", r"\bhgb\b"],
        # optional: vitals/procedures bucket if you want to reduce LAB_DATA__other size
        "vitals_procedures": [r"\bbp\b", r"\bhr\b", r"\bspo2\b", r"\bultrasound\b", r"\bct\b", r"\bmri\b", r"\bx[-\s]?ray\b"],
    },

    "OTHER_MEDS": {
        "anticoagulant": [
            r"\bwarfarin\b", r"\bheparin\b", r"\beliquis\b", r"\bxarelto\b",
            r"\bapixaban\b", r"\brivaroxaban\b"
        ],
        "statin": [r"\bstatin\b", r"\batorvastatin\b", r"\bsimvastatin\b", r"\brosuvastatin\b"],
        "immunosuppressant": [r"\bprednisone\b", r"\bsteroid\b", r"\bmethotrexate\b", r"\btacrolimus\b", r"\bcyclosporine\b"],
        "diabetes_meds": [r"\binsulin\b", r"\bmetformin\b", r"\bsemaglutide\b", r"\bozempic\b", r"\bglp[-\s]?1\b"],
        # NEW: thyroid meds commonly present in meds lists
        "thyroid_meds": [r"\blevothyroxine\b", r"\bsynthroid\b", r"\bliothyronine\b"],
        # NEW: antidepressants/anxiolytics common in VAERS meds lists
        "psych_meds": [r"\bsertraline\b", r"\bfluoxetine\b", r"\bcitalopram\b", r"\bescitalopram\b", r"\bbuspirone\b", r"\balprazolam\b"],
    },
}

def compile_map(category_map):
    compiled = {}
    for col, cats in category_map.items():
        compiled[col] = {cat: [re.compile(p) for p in pats] for cat, pats in cats.items()}
    return compiled

COMPILED_MAP = compile_map(CATEGORY_MAP)

def flag_any(text: str, patterns) -> int:
    if not text:
        return 0
    return int(any(p.search(text) for p in patterns))

# ----------------------------
# APPLY TO df_clean
# ----------------------------
df_clean = df_clean.copy()

for col in FREE_TEXT_FIELDS:
    if col not in df_clean.columns:
        continue

    norm_col = f"{col}_NORM"
    df_clean[norm_col] = df_clean[col].apply(normalize_text)

    # missingness indicator after null-token normalization
    df_clean[f"{col}_MISSING"] = (df_clean[norm_col].str.len() == 0).astype("int8")

    # dictionary flags
    if col in COMPILED_MAP:
        cat_cols = []
        for cat, pats in COMPILED_MAP[col].items():
            outcol = f"{col}__{cat}"
            df_clean[outcol] = df_clean[norm_col].apply(lambda t: flag_any(t, pats)).astype("int8")
            cat_cols.append(outcol)

        # "other" = has text but did not match any category
        df_clean[f"{col}__other"] = (
            (df_clean[f"{col}_MISSING"] == 0) & (df_clean[cat_cols].sum(axis=1) == 0)
        ).astype("int8")

# ----------------------------
# SAVE ENGINEERED FEATURES
# ----------------------------
engineered_cols = [c for c in df_clean.columns if "__" in c or c.endswith("_MISSING")]

comorb_df = df_clean[["VAERS_ID"] + engineered_cols].copy()

# 1) Always save CSV (no extra deps)
csv_path = os.path.join(OUTDIR, "comorbidity_indicators.csv")
comorb_df.to_csv(csv_path, index=False)
print("Saved CSV:", csv_path, "shape:", comorb_df.shape)

# 2) Optional Parquet (only if engine installed)
parquet_path = os.path.join(OUTDIR, "comorbidity_indicators.parquet")
try:
    comorb_df.to_parquet(parquet_path, index=False)
    print("Saved Parquet:", parquet_path)
except Exception as e:
    print("Parquet not saved (missing engine). CSV is saved and sufficient.")
    print("Parquet error:", repr(e))

# ----------------------------
# QUICK DIAGNOSTICS
# ----------------------------
print("\nIndicator count:", len(engineered_cols))
top_counts = comorb_df[engineered_cols].sum().sort_values(ascending=False).head(30)
diag_df = pd.DataFrame({"count": top_counts, "percent": (top_counts / len(comorb_df) * 100).round(3)})
print("\nTop indicators by prevalence:")
print(diag_df)

# Optional: show "OTHER" examples for each field to see what you're missing
def show_other_examples(field, n=10):
    other_col = f"{field}__other"
    if other_col not in df_clean.columns:
        return
    mask = (df_clean[other_col] == 1) & (df_clean[field].notna())
    print(f"\n{field} OTHER examples (n={n}):")
    print(df_clean.loc[mask, field].head(n))

for fld in ["HISTORY", "CUR_ILL", "ALLERGIES", "PRIOR_VAX", "LAB_DATA", "OTHER_MEDS"]:
    if fld in df_clean.columns:
        show_other_examples(fld, n=10)

Saved CSV: /Users/ariellerothman/Desktop/Capstone/Outputs/comorbidity_indicators.csv shape: (986096, 63)
Parquet not saved (missing engine). CSV is saved and sufficient.
Parquet error: ImportError("Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.\nA suitable version of pyarrow or fastparquet is required for parquet support.\nTrying to import the above resulted in these errors:\n - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.\n - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.")

Indicator count: 62

Top indicators by prevalence:
                                    count  percent
PRIOR_VAX_MISSING                  940654   95.392
CUR_ILL_MISSING                    862779   87.494
ALLERGIES_MISSING                  743721   75.421
LAB_DATA_MISSING                   726769   73.702
OTHER_MEDS_MISSING                 

In [4]:
# ============================================================
# 17. SYMPTOM TEXT PREPROCESSING
#     Updated supervised leakage cleanup with additional
#     quasi-administrative terms based on current model output
# ============================================================

import re
import pandas as pd

# --------------------------------------------------
# Base text cleaning
# --------------------------------------------------
_url_re = re.compile(r"http\S+|www\S+|https\S+")
_email_re = re.compile(r"\S+@\S+")
_num_re = re.compile(r"\d+")
_keep_re = re.compile(r"[^a-z0-9\s\-]")
_ws2 = re.compile(r"\s+")

def clean_symptom_text(x):
    if pd.isna(x) or x == "":
        return ""
    x = str(x).strip().lower()
    x = _url_re.sub(" ", x)
    x = _email_re.sub(" ", x)
    x = _num_re.sub(" __NUM__ ", x)
    x = _keep_re.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN"] = df_clean["SYMPTOM_TEXT"].apply(clean_symptom_text)

# --------------------------------------------------
# Strong supervised leakage scrubbing for severity prediction
# Removes direct outcome / adjudication / downstream-care terms
# plus residual administrative/report-style phrases
# --------------------------------------------------
LEAKAGE_PATTERNS = [re.compile(p) for p in [

    # ----------------------------------------------
    # Hospital / admission / downstream care
    # ----------------------------------------------
    r"\bhospital\w*\b",
    r"\badmit\w*\b",
    r"\binpatient\b",
    r"\bdischarg\w*\b",
    r"\bed\b",
    r"\ber\b",
    r"\bemergency\b",
    r"\bemergency room\b",
    r"\burgent\b",
    r"\burgent care\b",
    r"\btransport\w*\b",
    r"\btransfer\w*\b",
    r"\bambulance\b",
    r"\bems\b",
    r"\bhospice\b",

    # ----------------------------------------------
    # Death / life-threatening / rescue care
    # ----------------------------------------------
    r"\bdeath\b",
    r"\bdied\b",
    r"\bdead\b",
    r"\bfatal\b",
    r"\bpassed away\b",
    r"\bautopsy\b",
    r"\blife[-\s]?threaten\w*\b",
    r"\bintubat\w*\b",
    r"\bventilat\w*\b",
    r"\bicu\b",
    r"\barrest\b",

    # ----------------------------------------------
    # Seriousness / adjudication labels
    # ----------------------------------------------
    r"\bserious\w*\b",
    r"\bnon[-\s]?serious\w*\b",
    r"\bmedically significant\b",
    r"\bmedically important\b",
    r"\bcriterion\b",
    r"\bcriteria\b",
    r"\boutcome\b",
    r"\bdisability\b",
    r"\bpermanent\b",
    r"\bprolonged\b",

    # ----------------------------------------------
    # Administrative / report-template phrasing
    # ----------------------------------------------
    r"\breported cause\b",
    r"\breport medically\b",
    r"\bcriterion hospitalization\b",
    r"\bhospitalization medically\b",
    r"\bsignificant outcome\b",
    r"\bnarrative\b",
    r"\breport\b",
    r"\breported\b",
    r"\bspontaneous\b",
    r"\bcase\b",
    r"\bsource\b",
    r"\bpatient unknown\b",
    r"\bunknown\b",

    # ----------------------------------------------
    # Residual report-style / quasi-administrative phrases
    # seen in current model outputs
    # ----------------------------------------------
    r"\bevent resulted\b",
    r"\bevents resulted\b",
    r"\bresulted\b",
    r"\bcaused prolonged\b",
    r"\bmedical center\b",
    r"\bclinic care\b",
    r"\bwent care\b",
    r"\bvisited\b",
    r"\bperformed\b",

    # ----------------------------------------------
    # Other report-level phrasing seen previously
    # ----------------------------------------------
    r"\broom\b",
    r"\bdepartment\b",
    r"\bvisit\b",
    r"\bwent room\b",
    r"\bwent doctor\b",
    r"\bdoctor visit\b"
]]

def scrub_leakage_terms_strong(x):
    if pd.isna(x) or x == "":
        return ""
    x = str(x)
    for p in LEAKAGE_PATTERNS:
        x = p.sub(" ", x)
    x = _ws2.sub(" ", x).strip()
    return x

df_clean["SYMPTOM_TEXT_CLEAN_NOLEAK"] = (
    df_clean["SYMPTOM_TEXT_CLEAN"]
    .apply(scrub_leakage_terms_strong)
)

# --------------------------------------------------
# Clustering-specific administrative cleanup
# Keep separate from supervised no-leak field
# --------------------------------------------------
admin_phrases = [
    # vaccine / manufacturer boilerplate
    r"\bcovid vaccine\b",
    r"\bcovid immunisation\b",
    r"\bmrna moderna\b",
    r"\bmoderna covid\b",
    r"\bpfizer[-\s]?biontech\b",
    r"\bbiontech\b",
    r"\bmoderna\b",
    r"\bpfizer\b",
    r"\bjanssen\b",
    r"\bnovavax\b",
    r"\bbnt\b",
    r"\bmrna\b",
    r"\bvaccine\b",
    r"\bcovid\b",

    # lot / batch / product fields
    r"\blot number\b",
    r"\bbatch number\b",
    r"\bbatch lot\b",
    r"\bvaccine lot\b",
    r"\blot\b",
    r"\bbatch\b",
    r"\bdosage form\b",
    r"\bform\b",
    r"\broute administration\b",
    r"\broute admin\b",
    r"\bunspecified route\b",
    r"\badministration\b",
    r"\bintramuscular\b",
    r"\bsuspension injection\b",
    r"\bsuspension\b",

    # generic case-report boilerplate
    r"\bpatient received\b",
    r"\bpatient experienced\b",
    r"\bsubject experienced\b",
    r"\bspontaneous report\b",
    r"\bmedical history\b",
    r"\bmedical information\b",
    r"\bfollowing information\b",
    r"\bdescribes occurrence\b",
    r"\boccurrence\b",
    r"\bprovided\b",
    r"\bcontactable\b",
    r"\bcase\b",
    r"\breporter\b",
    r"\breported\b",
    r"\breport\b",
    r"\boutcome\b",
    r"\bunknown date\b",
    r"\bdate patient\b",
    r"\bdate\b",

    # dosing / process boilerplate
    r"\bsingle dose\b",
    r"\bdose\b",
    r"\bbooster\b",
    r"\badmin\b",
    r"\bprophylactic vaccination\b",
    r"\breceived\b",
    r"\bgiven\b",
    r"\badministered\b",
    r"\buse\b",

    # recurring vague metadata terms
    r"\bunknown\b",
    r"\binformation\b",
    r"\bdescribed\b",
    r"\bdescribes\b",
    r"\boccurred\b",
    r"\bpatient\b",
    r"\bsubject\b"
]

admin_regexes = [re.compile(p) for p in admin_phrases]

def remove_admin_terms(text):
    if pd.isna(text) or text == "":
        return ""
    text = str(text).lower()
    for rgx in admin_regexes:
        text = rgx.sub(" ", text)
    text = _ws2.sub(" ", text).strip()
    return text

df_clean["SYMPTOM_TEXT_CLEAN_CLUSTER"] = (
    df_clean["SYMPTOM_TEXT_CLEAN"]
    .apply(remove_admin_terms)
)

# --------------------------------------------------
# Optional token count after cluster cleanup
# --------------------------------------------------
df_clean["CLUSTER_TOKEN_COUNT"] = (
    df_clean["SYMPTOM_TEXT_CLEAN_CLUSTER"]
    .str.split()
    .str.len()
    .fillna(0)
    .astype(int)
)

# Optional clustering-specific subset
df_cluster = df_clean[df_clean["CLUSTER_TOKEN_COUNT"] >= 3].copy()

print("Rows in full dataset:", len(df_clean))
print("Rows retained for clustering:", len(df_cluster))
print("Rows dropped as near-empty after cluster cleanup:", len(df_clean) - len(df_cluster))

# --------------------------------------------------
# Save cleaned text variants
# --------------------------------------------------
df_clean[
    [
        "VAERS_ID",
        "SYMPTOM_TEXT",
        "SYMPTOM_TEXT_CLEAN",
        "SYMPTOM_TEXT_CLEAN_NOLEAK",
        "SYMPTOM_TEXT_CLEAN_CLUSTER",
        "CLUSTER_TOKEN_COUNT"
    ]
].to_csv(
    f"{OUTDIR}/symptom_text_clean_variants.csv",
    index=False
)

print([c for c in df_clean.columns if "SYMPTOM" in c or "CLUSTER_TOKEN_COUNT" in c])


Rows in full dataset: 986096
Rows retained for clustering: 955355
Rows dropped as near-empty after cluster cleanup: 30741
['SYMPTOM_TEXT', 'SYMPTOM_TEXT_CLEAN', 'SYMPTOM_TEXT_CLEAN_NOLEAK', 'SYMPTOM_TEXT_CLEAN_CLUSTER', 'CLUSTER_TOKEN_COUNT']


In [5]:
# ============================================================
# 24. SUPERVISED MODEL SETUP
#     Cleaned final version for severity prediction
# ============================================================

TEXT_FEATURE = "SYMPTOM_TEXT_CLEAN_NOLEAK"
y = df_clean["SERIOUS"].astype(int)

# --------------------------------------------------
# Numeric features
# --------------------------------------------------
# Keep onset-timing variables for the main model
# Remove HOSPDAYS because it is downstream / leakage-prone
numeric_cols = [
    c for c in ["AGE_YRS", "NUMDAYS", "ONSET_INTERVAL", "MAX_DOSE"]
    if c in df_clean.columns
]

# --------------------------------------------------
# Categorical features
# --------------------------------------------------
categorical_cols = [
    c for c in ["SEX", "STATE", "YEAR", "MONTH"]
    if c in df_clean.columns
]

# --------------------------------------------------
# Manufacturer and dose-related features
# --------------------------------------------------
manufacturer_cols = [c for c in df_clean.columns if c.startswith("MANU__")]

dose_cols = [
    c for c in ["DOSE_COUNT", "MULTI_DOSE", "UNKNOWN_DOSE", "MULTI_MANUFACTURER"]
    if c in df_clean.columns
]

# --------------------------------------------------
# Missingness indicators
# --------------------------------------------------
missing_cols = [
    c for c in [
        "NUMDAYS_MISSING",
        "ONSET_INTERVAL_MISSING",
        "AGE_YRS_MISSING",
        "HISTORY_MISSING",
        "CUR_ILL_MISSING",
        "ALLERGIES_MISSING",
        "PRIOR_VAX_MISSING",
        "LAB_DATA_MISSING",
        "OTHER_MEDS_MISSING"
    ]
    if c in df_clean.columns
]

# --------------------------------------------------
# Engineered clinical / comorbidity binary features
# --------------------------------------------------
comorb_prefixes = (
    "HISTORY__",
    "CUR_ILL__",
    "ALLERGIES__",
    "PRIOR_VAX__",
    "LAB_DATA__",
    "OTHER_MEDS__"
)

comorb_cols = [c for c in df_clean.columns if c.startswith(comorb_prefixes)]

# --------------------------------------------------
# Explicit exclusions for leakage-prone / helper / raw columns
# --------------------------------------------------
exclude_cols = {
    # target / direct leakage
    "SERIOUS",
    "DIED",
    "L_THREAT",
    "ER_VISIT",
    "ER_ED_VISIT",
    "HOSPITAL",
    "DISABLE",
    "BIRTH_DEFECT",
    "HOSPDAYS",
    "DATEDIED",

    # identifiers / raw dates
    "VAERS_ID",
    "RECVDATE",
    "VAX_DATE",
    "ONSET_DATE",

    # raw free-text fields (already engineered elsewhere)
    "SYMPTOM_TEXT",
    "LAB_DATA",
    "OTHER_MEDS",
    "CUR_ILL",
    "HISTORY",
    "PRIOR_VAX",
    "ALLERGIES",

    # normalized helper text columns
    "HISTORY_NORM",
    "CUR_ILL_NORM",
    "ALLERGIES_NORM",
    "PRIOR_VAX_NORM",
    "LAB_DATA_NORM",
    "OTHER_MEDS_NORM",

    # text/clustering helper columns not used as structured inputs
    "SYMPTOM_TEXT_CLEAN",
    "SYMPTOM_TEXT_CLEAN_CLUSTER",
    "CLUSTER_TOKEN_COUNT",

    # optional: usually unhelpful if dataset is already one vaccine type
    "VAX_TYPE"
}

# --------------------------------------------------
# Final structured feature list
# --------------------------------------------------
structured_cols = (
    numeric_cols +
    categorical_cols +
    manufacturer_cols +
    dose_cols +
    missing_cols +
    comorb_cols
)

# Remove anything explicitly excluded
structured_cols = [c for c in structured_cols if c not in exclude_cols]

# Remove duplicates while preserving order
structured_cols = list(dict.fromkeys(structured_cols))

# --------------------------------------------------
# Diagnostics
# --------------------------------------------------
print("TEXT_FEATURE:", TEXT_FEATURE)
print("Target:", "SERIOUS")
print()

print("Numeric cols:", numeric_cols)
print("Categorical cols:", categorical_cols)
print("Manufacturer cols:", manufacturer_cols)
print("Dose cols:", dose_cols)
print("Missing cols:", missing_cols)
print("Comorbidity/clinical cols count:", len(comorb_cols))
print("Total structured feature count:", len(structured_cols))
print()

print("Sample comorbidity features:")
print(comorb_cols[:20])
print()

print("First 60 structured cols:")
print(structured_cols[:60])


TEXT_FEATURE: SYMPTOM_TEXT_CLEAN_NOLEAK
Target: SERIOUS

Numeric cols: ['AGE_YRS', 'NUMDAYS', 'ONSET_INTERVAL', 'MAX_DOSE']
Categorical cols: ['SEX', 'STATE', 'YEAR', 'MONTH']
Manufacturer cols: ['MANU__JANSSEN', 'MANU__MODERNA', 'MANU__NOVAVAX', 'MANU__PFIZER\\BIONTECH', 'MANU__UNKNOWN MANUFACTURER']
Dose cols: ['DOSE_COUNT', 'MULTI_DOSE', 'UNKNOWN_DOSE', 'MULTI_MANUFACTURER']
Missing cols: ['NUMDAYS_MISSING', 'ONSET_INTERVAL_MISSING', 'AGE_YRS_MISSING', 'HISTORY_MISSING', 'CUR_ILL_MISSING', 'ALLERGIES_MISSING', 'PRIOR_VAX_MISSING', 'LAB_DATA_MISSING', 'OTHER_MEDS_MISSING']
Comorbidity/clinical cols count: 48
Total structured feature count: 74

Sample comorbidity features:
['HISTORY__cardiovascular', 'HISTORY__respiratory', 'HISTORY__metabolic', 'HISTORY__endocrine_thyroid', 'HISTORY__musculoskeletal', 'HISTORY__mental_health', 'HISTORY__gastrointestinal', 'HISTORY__kidney', 'HISTORY__clotting', 'HISTORY__autoimmune', 'HISTORY__cancer_immunocompromised', 'HISTORY__neurologic', 'HISTOR

In [ ]:
import pandas as pd

print("Total columns:", len(df_clean.columns))

pd.set_option("display.max_rows", None)

col_df = pd.DataFrame({
    "column": df_clean.columns,
    "dtype": df_clean.dtypes
})

display(col_df)

In [6]:
# ============================================================
# 25. PREPROCESSORS + MODELS
#    Raw TF-IDF version (NO SVD)
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, MaxAbsScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# --------------------------------------------------
# Structured preprocessing
# --------------------------------------------------
numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

structured_binary_cols = [c for c in structured_cols if c not in numeric_cols + categorical_cols]

structured_preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, numeric_cols),
        ("cat", categorical_pipe, categorical_cols),
        ("passthrough_binary", "passthrough", structured_binary_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# --------------------------------------------------
# Raw TF-IDF preprocessing (NO SVD)
# --------------------------------------------------
text_preprocess = Pipeline([
    ("tfidf", TfidfVectorizer(
        max_features=5000,          # you can lower to 3000 if runtime is too heavy
        ngram_range=(1, 2),         # can switch to (1,1) if needed for speed
        min_df=10,
        max_df=0.9,
        stop_words="english",
        sublinear_tf=True,
        norm="l2",
        strip_accents="unicode",
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z\-]+\b|__NUM__"
    ))
])

# --------------------------------------------------
# Combine structured + raw TF-IDF text
# --------------------------------------------------
full_preprocess = ColumnTransformer(
    transformers=[
        ("structured", structured_preprocess, structured_cols),
        ("text", text_preprocess, TEXT_FEATURE),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

# Optional scaling for sparse matrix models
# MaxAbsScaler works well with sparse matrices
# Especially useful for logistic regression
raw_tfidf_pipeline_template = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression())   # placeholder; will be replaced per model
])

# --------------------------------------------------
# Models compatible with sparse raw TF-IDF
# --------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42
    ),
    "Decision Tree": DecisionTreeClassifier(
        class_weight="balanced",
        random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )
}

# --------------------------------------------------
# Parameter grids
# --------------------------------------------------
param_grids = {
    "Logistic Regression": {
        "clf__C": [0.1, 1.0, 3.0]
    },
    "Decision Tree": {
        "clf__max_depth": [5, 10, 20],
        "clf__min_samples_split": [2, 5, 10]
    },
    "Random Forest": {
        "clf__n_estimators": [100, 200],
        "clf__max_depth": [10, 20],
        "clf__min_samples_split": [2, 5]
    }
}

print("Models to run:", list(models.keys()))

Models to run: ['Logistic Regression', 'Decision Tree', 'Random Forest']


In [7]:
# ============================================================
# 26. LIGHTER STRATIFIED K-FOLD CV + GRID SEARCH
#    Raw TF-IDF version
# ============================================================

from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict, train_test_split
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score, confusion_matrix
import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Stratified subset for tuning
# --------------------------------------------------
subset_size = min(100000, len(df_clean))

df_tune, _ = train_test_split(
    df_clean,
    train_size=subset_size,
    stratify=df_clean["SERIOUS"],
    random_state=42
)

y_tune = df_tune["SERIOUS"].astype(int)

print("Tuning subset shape:", df_tune.shape)
print("Serious prevalence in tuning subset:", y_tune.mean())

# --------------------------------------------------
# 2. CV setup
# --------------------------------------------------
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

results = []
best_models = {}

# --------------------------------------------------
# 3. Lighter parameter grids
# --------------------------------------------------
light_param_grids = {
    "Logistic Regression": {
        "clf__C": [0.1, 1.0]
    },
    "Decision Tree": {
        "clf__max_depth": [10, 20],
        "clf__min_samples_split": [5, 10]
    },
    "Random Forest": {
        "clf__n_estimators": [100],
        "clf__max_depth": [10, 20]
    }
}

# --------------------------------------------------
# 4. Tune and evaluate
# --------------------------------------------------
for name, clf in models.items():
    print("\n" + "=" * 80)
    print("MODEL:", name)
    print("=" * 80)

    pipe = Pipeline([
        ("preprocess", full_preprocess),
        ("scale", MaxAbsScaler()),
        ("clf", clf)
    ])

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=light_param_grids[name],
        scoring="average_precision",
        cv=cv,
        n_jobs=1,      # safer for laptops
        verbose=1
    )

    grid.fit(df_tune, y_tune)
    best = grid.best_estimator_
    best_models[name] = best

    print("Best params:", grid.best_params_)

    # Out-of-fold predictions on tuning subset
    y_pred = cross_val_predict(best, df_tune, y_tune, cv=cv, method="predict", n_jobs=1)

    # Most classifiers here support predict_proba
    y_prob = cross_val_predict(best, df_tune, y_tune, cv=cv, method="predict_proba", n_jobs=1)[:, 1]

    pr_auc = average_precision_score(y_tune, y_prob)
    f1 = f1_score(y_tune, y_pred, zero_division=0)
    precision = precision_score(y_tune, y_pred, zero_division=0)
    recall = recall_score(y_tune, y_pred, zero_division=0)
    cm = confusion_matrix(y_tune, y_pred)

    results.append({
        "Model": name,
        "PR-AUC": pr_auc,
        "F1": f1,
        "Precision": precision,
        "Recall": recall,
        "BestParams": grid.best_params_,
        "ConfusionMatrix": cm
    })

    print(f"PR-AUC:    {pr_auc:.4f}")
    print(f"F1:        {f1:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print("Confusion Matrix:")
    print(cm)

summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k != "ConfusionMatrix"}
    for r in results
])

print("\nMODEL COMPARISON")
print(summary_df.sort_values("PR-AUC", ascending=False))

summary_df.to_csv(f"{OUTDIR}/supervised_model_comparison_raw_tfidf.csv", index=False)

# Save best models in memory for later feature importance extraction
print("\nStored best models:", list(best_models.keys()))


Tuning subset shape: (100000, 106)
Serious prevalence in tuning subset: 0.19851

MODEL: Logistic Regression
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best params: {'clf__C': 1.0}
PR-AUC:    0.7914
F1:        0.6986
Precision: 0.6073
Recall:    0.8221
Confusion Matrix:
[[69596 10553]
 [ 3531 16320]]

MODEL: Decision Tree
Fitting 3 folds for each of 4 candidates, totalling 12 fits
Best params: {'clf__max_depth': 10, 'clf__min_samples_split': 10}
PR-AUC:    0.6206
F1:        0.6091
Precision: 0.5090
Recall:    0.7584
Confusion Matrix:
[[65625 14524]
 [ 4796 15055]]

MODEL: Random Forest
Fitting 3 folds for each of 2 candidates, totalling 6 fits
Best params: {'clf__max_depth': 20, 'clf__n_estimators': 100}
PR-AUC:    0.7316
F1:        0.6432
Precision: 0.5558
Recall:    0.7632
Confusion Matrix:
[[68042 12107]
 [ 4700 15151]]

MODEL COMPARISON
                 Model    PR-AUC        F1  Precision    Recall  \
0  Logistic Regression  0.791369  0.698570   0.607301  0.822125  

In [8]:
# ============================================================
# 27. OPTIONAL COMPARISON:
#     STRUCTURED-ONLY vs STRUCTURED + RAW TF-IDF
# ============================================================

from sklearn.model_selection import cross_val_predict
from sklearn.metrics import average_precision_score, f1_score, precision_score, recall_score

comparison_results = []

# --------------------------------------------------
# Structured-only pipeline
# --------------------------------------------------
structured_only_preprocess = ColumnTransformer(
    transformers=[
        ("structured", structured_preprocess, structured_cols),
    ],
    remainder="drop",
    sparse_threshold=1.0
)

structured_only_pipe = Pipeline([
    ("preprocess", structured_only_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42,
        C=1.0
    ))
])

# --------------------------------------------------
# Structured + text pipeline
# --------------------------------------------------
structured_text_pipe = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42,
        C=1.0
    ))
])

for label, pipe in {
    "Structured only": structured_only_pipe,
    "Structured + raw TF-IDF": structured_text_pipe
}.items():
    print("\nRunning:", label)

    y_pred = cross_val_predict(pipe, df_tune, y_tune, cv=cv, method="predict", n_jobs=1)
    y_prob = cross_val_predict(pipe, df_tune, y_tune, cv=cv, method="predict_proba", n_jobs=1)[:, 1]

    comparison_results.append({
        "FeatureSet": label,
        "PR-AUC": average_precision_score(y_tune, y_prob),
        "F1": f1_score(y_tune, y_pred, zero_division=0),
        "Precision": precision_score(y_tune, y_pred, zero_division=0),
        "Recall": recall_score(y_tune, y_pred, zero_division=0)
    })

comparison_df = pd.DataFrame(comparison_results)
print("\nSTRUCTURED vs TEXT COMPARISON")
print(comparison_df.sort_values("PR-AUC", ascending=False))

comparison_df.to_csv(f"{OUTDIR}/structured_vs_raw_tfidf_comparison.csv", index=False)



Running: Structured only

Running: Structured + raw TF-IDF

STRUCTURED vs TEXT COMPARISON
                FeatureSet    PR-AUC        F1  Precision    Recall
1  Structured + raw TF-IDF  0.791369  0.698570   0.607301  0.822125
0          Structured only  0.542422  0.529115   0.430746  0.685709


In [9]:
# ============================================================
# 28. FIT FINAL LOGISTIC REGRESSION MODEL
#     FOR FEATURE IMPORTANCE
# ============================================================

final_logreg_pipe = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", LogisticRegression(
        max_iter=3000,
        class_weight="balanced",
        solver="saga",
        n_jobs=-1,
        random_state=42,
        C=1.0
    ))
])

final_logreg_pipe.fit(df_tune, y_tune)

print("Final logistic regression model fitted on tuning subset.")


Final logistic regression model fitted on tuning subset.


In [10]:
# ============================================================
# 29. EXTRACT TOP FEATURES FROM LOGISTIC REGRESSION
#     (structured + raw TF-IDF)
# ============================================================

import numpy as np
import pandas as pd

# --------------------------------------------------
# Get fitted preprocessors
# --------------------------------------------------
preprocessor = final_logreg_pipe.named_steps["preprocess"]
clf = final_logreg_pipe.named_steps["clf"]

# Structured feature names
structured_feature_names = preprocessor.named_transformers_["structured"].get_feature_names_out()

# Text feature names
text_feature_names = preprocessor.named_transformers_["text"].named_steps["tfidf"].get_feature_names_out()

# Combine all feature names
all_feature_names = np.concatenate([structured_feature_names, text_feature_names])

# Logistic regression coefficients
coefs = clf.coef_[0]

coef_df = pd.DataFrame({
    "feature": all_feature_names,
    "coefficient": coefs,
    "abs_coefficient": np.abs(coefs),
    "feature_type": ["structured"] * len(structured_feature_names) + ["text"] * len(text_feature_names)
})

# Top positive predictors of seriousness
top_positive = coef_df.sort_values("coefficient", ascending=False).head(50)

# Top negative predictors of seriousness
top_negative = coef_df.sort_values("coefficient", ascending=True).head(50)

# Top text-only predictors
top_text_positive = coef_df[coef_df["feature_type"] == "text"].sort_values("coefficient", ascending=False).head(50)
top_text_negative = coef_df[coef_df["feature_type"] == "text"].sort_values("coefficient", ascending=True).head(50)

print("\nTop positive predictors of SERIOUS=1")
print(top_positive[["feature", "coefficient", "feature_type"]])

print("\nTop negative predictors of SERIOUS=1")
print(top_negative[["feature", "coefficient", "feature_type"]])

print("\nTop POSITIVE text predictors")
print(top_text_positive[["feature", "coefficient"]])

print("\nTop NEGATIVE text predictors")
print(top_text_negative[["feature", "coefficient"]])

# Save results
coef_df.to_csv(f"{OUTDIR}/logreg_all_feature_coefficients_raw_tfidf.csv", index=False)
top_positive.to_csv(f"{OUTDIR}/logreg_top_positive_features_raw_tfidf.csv", index=False)
top_negative.to_csv(f"{OUTDIR}/logreg_top_negative_features_raw_tfidf.csv", index=False)
top_text_positive.to_csv(f"{OUTDIR}/logreg_top_positive_text_features_raw_tfidf.csv", index=False)
top_text_negative.to_csv(f"{OUTDIR}/logreg_top_negative_text_features_raw_tfidf.csv", index=False)



Top positive predictors of SERIOUS=1
              feature  coefficient feature_type
755              care     8.446654         text
773             cause     5.057333         text
267         admission     4.918296         text
2455               iv     4.886310         text
4437           stroke     4.646434         text
5072             went     4.283950         text
1529              epi     4.060732         text
758      care patient     3.955221         text
1530      epinephrine     3.939488         text
5075        went care     3.781551         text
751           cardiac     3.673530         text
856       clinic care     3.643924         text
1531           epipen     3.579920         text
3520        pneumonia     3.539700         text
373      appendicitis     3.526468         text
5060  weeks receiving     3.350381         text
3567        presented     3.337829         text
3443           pepcid     3.337239         text
4143             sent     3.280650         text
35

In [11]:
# ============================================================
# 30. RANDOM FOREST FEATURE IMPORTANCE
# ============================================================

# Fit final random forest model
final_rf_pipe = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", RandomForestClassifier(
        n_estimators=100,
        max_depth=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

final_rf_pipe.fit(df_tune, y_tune)

preprocessor_rf = final_rf_pipe.named_steps["preprocess"]
rf = final_rf_pipe.named_steps["clf"]

structured_feature_names_rf = preprocessor_rf.named_transformers_["structured"].get_feature_names_out()
text_feature_names_rf = preprocessor_rf.named_transformers_["text"].named_steps["tfidf"].get_feature_names_out()
all_feature_names_rf = np.concatenate([structured_feature_names_rf, text_feature_names_rf])

rf_importance_df = pd.DataFrame({
    "feature": all_feature_names_rf,
    "importance": rf.feature_importances_,
    "feature_type": ["structured"] * len(structured_feature_names_rf) + ["text"] * len(text_feature_names_rf)
}).sort_values("importance", ascending=False)

print("\nTop Random Forest features")
print(rf_importance_df.head(50))

rf_importance_df.to_csv(f"{OUTDIR}/random_forest_feature_importance_raw_tfidf.csv", index=False)



Top Random Forest features
                                              feature  importance feature_type
1                                        num__NUMDAYS    0.052629   structured
100              passthrough_binary__LAB_DATA_MISSING    0.051163   structured
2                                 num__ONSET_INTERVAL    0.043424   structured
141   passthrough_binary__LAB_DATA__vitals_procedures    0.021979   structured
4236                                             site    0.014407         text
0                                        num__AGE_YRS    0.013020   structured
142               passthrough_binary__LAB_DATA__other    0.012308   structured
806                                             chest    0.011453         text
1048                                            covid    0.009168         text
202                                             acute    0.008246         text
3395                                 patient received    0.008204         text
89                     p

In [12]:
# ============================================================
# 31. DECISION TREE FEATURE IMPORTANCE
#     (structured + raw TF-IDF)
# ============================================================

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import MaxAbsScaler

# --------------------------------------------------
# Fit final decision tree model
# --------------------------------------------------
final_dt_pipe = Pipeline([
    ("preprocess", full_preprocess),
    ("scale", MaxAbsScaler()),
    ("clf", DecisionTreeClassifier(
        max_depth=20,              # adjust based on your chosen best params
        min_samples_split=5,
        class_weight="balanced",
        random_state=42
    ))
])

final_dt_pipe.fit(df_tune, y_tune)

print("Final decision tree model fitted.")


Final decision tree model fitted.


In [13]:
# ============================================================
# 32. EXTRACT DECISION TREE FEATURE IMPORTANCE
# ============================================================

# Get fitted objects
preprocessor_dt = final_dt_pipe.named_steps["preprocess"]
dt = final_dt_pipe.named_steps["clf"]

# Structured feature names
structured_feature_names_dt = preprocessor_dt.named_transformers_["structured"].get_feature_names_out()

# Text feature names
text_feature_names_dt = preprocessor_dt.named_transformers_["text"].named_steps["tfidf"].get_feature_names_out()

# Combine feature names
all_feature_names_dt = np.concatenate([structured_feature_names_dt, text_feature_names_dt])

# Importance values
dt_importances = dt.feature_importances_

# Put into dataframe
dt_importance_df = pd.DataFrame({
    "feature": all_feature_names_dt,
    "importance": dt_importances,
    "feature_type": ["structured"] * len(structured_feature_names_dt) + ["text"] * len(text_feature_names_dt)
})

# Sort descending
dt_importance_df = dt_importance_df.sort_values("importance", ascending=False)

# Top overall features
top_dt_features = dt_importance_df.head(50)

# Top text features only
top_dt_text_features = (
    dt_importance_df[dt_importance_df["feature_type"] == "text"]
    .sort_values("importance", ascending=False)
    .head(50)
)

# Top structured features only
top_dt_structured_features = (
    dt_importance_df[dt_importance_df["feature_type"] == "structured"]
    .sort_values("importance", ascending=False)
    .head(50)
)

print("\nTop overall Decision Tree features")
print(top_dt_features)

print("\nTop text Decision Tree features")
print(top_dt_text_features)

print("\nTop structured Decision Tree features")
print(top_dt_structured_features)

# Save results
dt_importance_df.to_csv(f"{OUTDIR}/decision_tree_feature_importance_raw_tfidf.csv", index=False)
top_dt_features.to_csv(f"{OUTDIR}/decision_tree_top_features_raw_tfidf.csv", index=False)
top_dt_text_features.to_csv(f"{OUTDIR}/decision_tree_top_text_features_raw_tfidf.csv", index=False)
top_dt_structured_features.to_csv(f"{OUTDIR}/decision_tree_top_structured_features_raw_tfidf.csv", index=False)



Top overall Decision Tree features
                                              feature  importance feature_type
1                                        num__NUMDAYS    0.224210   structured
100              passthrough_binary__LAB_DATA_MISSING    0.167037   structured
947                                       concomitant    0.052903         text
142               passthrough_binary__LAB_DATA__other    0.026843   structured
734                                            called    0.026464         text
0                                        num__AGE_YRS    0.022627   structured
755                                              care    0.018767         text
806                                             chest    0.015091         text
2455                                               iv    0.014225         text
202                                             acute    0.011534         text
773                                             cause    0.011379         text
1557            

In [14]:
 # ============================================================
# 33. FEATURES ACTUALLY USED IN THE DECISION TREE
# ============================================================

used_feature_idx = dt.tree_.feature
used_feature_idx = used_feature_idx[used_feature_idx >= 0]   # remove leaf markers (-2)

used_feature_names = all_feature_names_dt[used_feature_idx]
used_feature_names_unique = pd.Series(used_feature_names).value_counts().reset_index()
used_feature_names_unique.columns = ["feature", "num_splits"]

print("\nFeatures actually used in tree splits")
print(used_feature_names_unique.head(50))

used_feature_names_unique.to_csv(
    f"{OUTDIR}/decision_tree_features_used_in_splits_raw_tfidf.csv",
    index=False
)


Features actually used in tree splits
                                   feature  num_splits
0                             num__AGE_YRS          41
1                             num__NUMDAYS          19
2                      num__ONSET_INTERVAL          17
3                                  patient          15
4                                   doctor          14
5                                      arm          13
6                                     pain          13
7                                     days          13
8                                    covid          12
9                                     dose          12
10                                    care          11
11                                    went          11
12                               injection          11
13                                  better          11
14                                     day          10
15                             concomitant          10
16                        

In [15]:
# ============================================================
# 34. EXPORT DECISION TREE RULES
# ============================================================

from sklearn.tree import export_text

tree_rules = export_text(
    dt,
    feature_names=list(all_feature_names_dt),
    max_depth=4
)

print(tree_rules)

with open(f"{OUTDIR}/decision_tree_rules_raw_tfidf.txt", "w") as f:
    f.write(tree_rules)


|--- num__NUMDAYS <= 0.00
|   |--- passthrough_binary__LAB_DATA_MISSING <= 0.50
|   |   |--- passthrough_binary__LAB_DATA__other <= 0.50
|   |   |   |--- delivery <= 0.04
|   |   |   |   |--- appropriate section <= 0.39
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |   |--- appropriate section >  0.39
|   |   |   |   |   |--- truncated branch of depth 4
|   |   |   |--- delivery >  0.04
|   |   |   |   |--- num__AGE_YRS <= 0.25
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- num__AGE_YRS >  0.25
|   |   |   |   |   |--- truncated branch of depth 2
|   |   |--- passthrough_binary__LAB_DATA__other >  0.50
|   |   |   |--- concomitant <= 0.26
|   |   |   |   |--- arm <= 0.08
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |   |--- arm >  0.08
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |--- concomitant >  0.26
|   |   |   |   |--- care <= 0.03
|   |   |   |   |   |--- truncated branch of depth 16
|   |   |   |   |--- care 